# ContractSLM-3B — Training on Google Colab A100

Fine-tunes Llama-3.2-3B-Instruct on CUAD (real commercial contracts) + synthetic data.

**Runtime:** A100 GPU (Runtime → Change runtime type → A100)  
**Time:** ~3 hours  
**Cost:** Free with Colab Pro, or use a Colab Pro+ A100 session

**Note:** This is a 3B model — T4 (16GB) will work but slowly. A100 (40GB) is recommended.

In [ ]:
# 1. Check GPU
!nvidia-smi
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 2. Install
!pip install -q transformers datasets peft trl bitsandbytes accelerate \
               huggingface_hub pdfplumber python-docx fastapi uvicorn pydantic python-multipart
print('Done')

In [ ]:
# 3. HuggingFace login (needs access to Llama-3.2-3B-Instruct)
from huggingface_hub import login
login()

In [ ]:
# 4. Clone repo
!git clone https://github.com/YOUR_USERNAME/contract-slm.git
%cd contract-slm

In [ ]:
# 5. Build dataset — loads CUAD from HF + generates synthetic contracts (~3 min)
!python build_dataset.py
!echo 'Train:' && wc -l data/contract_train.jsonl
!echo 'Eval:'  && wc -l data/contract_eval.jsonl

In [ ]:
# 6. Train (~3 hours on A100, ~6 hours on T4)
!python train.py

In [ ]:
# 7. Evaluate
!python evaluate.py --model ./contract-slm-3b --n 100

In [ ]:
# 8. Quick inference test
import sys
sys.path.append('.')
from inference import ContractSLM, print_result

cs = ContractSLM('./contract-slm-3b')

test_contract = """
SOFTWARE AS A SERVICE AGREEMENT

This Agreement is entered into as of January 1, 2026 between Acme Corp ("Customer") 
and CloudSoft Inc ("Vendor").

LIMITATION OF LIABILITY
Vendor's total liability shall not exceed the fees paid by Customer in the one (1) month 
preceding the claim giving rise to liability.

RENEWAL
This Agreement will automatically renew for successive one-year terms unless Customer 
provides written notice of non-renewal at least ninety (90) days prior to the end of 
the then-current term.

DATA
Vendor may use Customer Data to improve its products and services and may share 
aggregated, anonymized data with third parties.

INDEMNIFICATION
Customer shall indemnify, defend and hold harmless Vendor from any claims arising 
from Customer's use of the Service, including claims arising from Vendor's negligence.
"""

result = cs.analyse(test_contract)
print_result(result)

In [ ]:
# 9. Push to HuggingFace + get SQL for marketplace
HF_USERNAME = 'your-username'   # change this
HF_TOKEN    = 'your-token'      # HF write token

!python push_to_hub.py --hf-token {HF_TOKEN} --username {HF_USERNAME}